# monomech modular notebook

This notebook shows how to use each modular stage and the full pipeline wrapper.


## 1. Setup


In [ ]:
import monomech as mm
from monomech.notebook import display_stage_summary, display_stage_table

VIDEO_PATH = 'subject01.mp4'
MODEL_PATH = 'pose_landmarker_heavy.task'
OSIM_MODEL = 'GATMA_Model.osim'


## 2. Create a trial


In [ ]:
trial = mm.Trial.from_video(VIDEO_PATH)
trial.video_metadata


## 3. pose2d


In [ ]:
pose2d = mm.pose2d.process(
    trial,
    config=mm.MediaPipePoseConfig(model_asset_path=MODEL_PATH, target_fps=60),
)
display_stage_summary(pose2d)
display_stage_table(pose2d, 'landmarks_long')
pose2d.figures['pose2d']


## 4. world3d


In [ ]:
world3d = mm.world3d.process(trial, pose2d=pose2d)
display_stage_summary(world3d)
display_stage_table(world3d, 'world3d_long')
display_stage_table(world3d, 'segment_lengths')
world3d.figures['world3d']


## 5. pnp


In [ ]:
pnp = mm.pnp.solve(trial, pose2d=pose2d, world3d=world3d)
display_stage_summary(pnp)
display_stage_table(pnp, 'camera_pose')
display_stage_table(pnp, 'reprojection')
pnp.figures['pnp_pose']


## 6. global_pose


In [ ]:
global_pose = mm.global_pose.estimate(
    trial,
    pose2d=pose2d,
    world3d=world3d,
    pnp=pnp,
)
display_stage_summary(global_pose)
display_stage_table(global_pose, 'global_pose_long')
display_stage_table(global_pose, 'contacts')
display_stage_table(global_pose, 'root_trajectory')
global_pose.figures['global_pose']


## 7. forces


In [ ]:
force_set = mm.ForceSet([
    mm.ExternalForce.constant(
        name='right_grf',
        target='right_foot',
        magnitude=900.0,
        direction=(0.0, 1.0, 0.0),
        point='right_ankle',
    ),
    mm.ExternalForce.constant(
        name='left_hand_load',
        target='left_hand',
        magnitude=75.0,
        direction=(0.0, -1.0, 0.0),
        point='left_wrist',
    ),
])
force_result = mm.forces.build(trial, global_pose=global_pose, force_set=force_set)
display_stage_summary(force_result)
display_stage_table(force_result, 'forces_long')
display_stage_table(force_result, 'mapping')


## 8. optional OpenSim IK / ID


In [ ]:
# Requires the OpenSim Python bindings in your environment.
# ik = mm.opensim.run_ik(trial, global_pose=global_pose, model_path=OSIM_MODEL)
# id_result = mm.opensim.run_id(trial, global_pose=global_pose, forces=force_result, model_path=OSIM_MODEL)


## 9. full pipeline wrapper


In [ ]:
pipeline = mm.FullPipeline(
    config=mm.PipelineConfig(
        pose=mm.MediaPipePoseConfig(model_asset_path=MODEL_PATH, target_fps=60),
        opensim=mm.OpenSimConfig(model_path=OSIM_MODEL, run_ik=False, run_id=False),
    ),
    stages=mm.PipelineStages(
        pose2d=True,
        world3d=True,
        pnp=True,
        global_pose=True,
        forces=False,
        ik=False,
        id=False,
    ),
)
run = pipeline.run(VIDEO_PATH, output_dir='outputs/subject01')
run.available_stages()


In [ ]:
run.pose2d.tables['summary']
run.world3d.tables['segment_lengths'].head()
run.pnp.tables['camera_pose'].head()
run.global_pose.tables['contacts'].head()
